# ELAB Autoresearch — Loop Parallelo con Kimi K2.5

**Ispirato a [github.com/karpathy/autoresearch](https://github.com/karpathy/autoresearch)**

Questo notebook gira su Colab in parallelo all'orchestratore locale.
Usa Kimi K2.5 come cervello di ricerca autonomo per studiare ELAB Tutor
da ogni angolo e produrre findings azionabili.

**Pattern**: modifica ipotesi → esperimento (ricerca) → misura → keep/discard

**Principio Zero**: L'insegnante inesperto è il vero utente.

---

© Andrea Marro — ELAB Tutor Autoresearch

In [ ]:
#@title 1. Setup — API Keys e Config
#@markdown Inserisci la tua Kimi API key (Moonshot AI)

KIMI_API_KEY = ''  #@param {type:"string"}
DEEPSEEK_API_KEY = ''  #@param {type:"string"}

# Config
MAX_ITERATIONS = 100  #@param {type:"integer"}
SECONDS_PER_ITERATION = 120  #@param {type:"integer"}
SYNC_TO_DRIVE = True  #@param {type:"boolean"}

import os
os.environ['KIMI_API_KEY'] = KIMI_API_KEY
os.environ['DEEPSEEK_API_KEY'] = DEEPSEEK_API_KEY

print('Setup completato.')

In [ ]:
#@title 2. Core — API Wrappers (immutabile, come prepare.py di Karpathy)

import json
import time
import re
import urllib.request
import urllib.error
from datetime import datetime
from pathlib import Path

MAX_RETRIES = 3
RETRY_DELAY = 5


def _api_call_with_retry(url, payload, headers, timeout=90):
    """HTTP POST with retry logic for resilience on Colab."""
    data = json.dumps(payload).encode('utf-8')
    for attempt in range(MAX_RETRIES):
        req = urllib.request.Request(url, data=data, headers=headers, method='POST')
        try:
            with urllib.request.urlopen(req, timeout=timeout) as resp:
                return json.loads(resp.read().decode('utf-8'))
        except (urllib.error.URLError, urllib.error.HTTPError, TimeoutError) as e:
            if attempt < MAX_RETRIES - 1:
                wait = RETRY_DELAY * (attempt + 1)
                print(f'    Retry {attempt+1}/{MAX_RETRIES} in {wait}s ({e})')
                time.sleep(wait)
            else:
                return {'error': str(e)}
        except Exception as e:
            return {'error': str(e)}
    return {'error': 'max retries exceeded'}


def call_kimi(prompt: str, max_tokens: int = 2048) -> str:
    """Call Kimi K2.5 via Moonshot API with retry."""
    key = os.environ.get('KIMI_API_KEY', '')
    if not key:
        return '[SKIP] No Kimi API key'

    resp = _api_call_with_retry(
        'https://api.moonshot.ai/v1/chat/completions',
        {'model': 'kimi-k2-0711', 'messages': [{'role': 'user', 'content': prompt}], 'max_tokens': max_tokens},
        {'Content-Type': 'application/json', 'Authorization': f'Bearer {key}'},
        timeout=90,
    )
    if 'error' in resp:
        return f'[ERROR] {resp["error"]}'
    try:
        return resp['choices'][0]['message']['content']
    except (KeyError, IndexError):
        return f'[ERROR] Unexpected response'


def call_deepseek(prompt: str, max_tokens: int = 1024) -> str:
    """Call DeepSeek-chat (fast, cheap) for scoring."""
    key = os.environ.get('DEEPSEEK_API_KEY', '')
    if not key:
        return '[SKIP] No DeepSeek API key'

    resp = _api_call_with_retry(
        'https://api.deepseek.com/v1/chat/completions',
        {'model': 'deepseek-chat', 'messages': [{'role': 'user', 'content': prompt}], 'max_tokens': max_tokens},
        {'Content-Type': 'application/json', 'Authorization': f'Bearer {key}'},
        timeout=60,
    )
    if 'error' in resp:
        return f'[ERROR] {resp["error"]}'
    try:
        return resp['choices'][0]['message']['content']
    except (KeyError, IndexError):
        return f'[ERROR] Unexpected response'


def parse_structured_output(text):
    """Robust parser for LLM structured output. Handles markdown, extra spaces, etc."""
    result = {'severity': 'low', 'evidence': 'speculation', 'verdict': 'DISCARD', 'action': 'none', 'finding': '', 'cov': ''}
    patterns = {
        'severity': r'(?:\*\*)?SEVERITY(?:\*\*)?[:\s]+(\w+)',
        'evidence': r'(?:\*\*)?EVIDENCE(?:\*\*)?[:\s]+(\w+)',
        'verdict': r'(?:\*\*)?VERDICT(?:\*\*)?[:\s]+(KEEP|DISCARD)',
        'action': r'(?:\*\*)?ACTION(?:\*\*)?[:\s]+(.+?)(?:\n|$)',
        'finding': r'(?:\*\*)?FINDING(?:\*\*)?[:\s]+(.+?)(?=\n(?:\*\*)?(?:EVIDENCE|SEVERITY|ACTION|COV|VERDICT)|$)',
        'cov': r'(?:\*\*)?COV(?:\*\*)?[:\s]+(.+?)(?:\n|$)',
    }
    for key, pattern in patterns.items():
        match = re.search(pattern, text, re.IGNORECASE | re.DOTALL)
        if match:
            val = match.group(1).strip()
            if key in ('severity', 'evidence'):
                val = val.lower()
            elif key == 'verdict':
                val = val.upper()
            result[key] = val
    return result


print('API wrappers pronti (con retry + parsing robusto).')

In [ ]:
#@title 3. program.md — Istruzioni (immutabile, come program.md di Karpathy)

PROGRAM_MD = """
# ELAB Autoresearch — Programma per Kimi K2.5

## PRINCIPIO ZERO
L'insegnante inesperto e' il vero utente di ELAB Tutor.
Galileo e' un libro intelligente, non un professore.
Tutti possono insegnare con ELAB Tutor.

## COSA E' ELAB TUTOR
- Tutor educativo per elettronica e Arduino, bambini 8-12 anni
- Simulatore di circuiti proprietario (MNA/KCL solver + avr8js)
- AI tutor 'Galileo' (5 specialisti via Anthropic API)
- 62 esperimenti in 3 volumi progressivi
- Target: scuole medie italiane, LIM, tablet
- Deploy: Vercel (www.elabtutor.school)

## REGOLE DI RICERCA
1. Ogni finding deve essere AZIONABILE (task concreto) o DECISIONALE (scelta informata)
2. Niente report ornamentali. Niente 'sembra interessante'.
3. Severity obbligatoria: blocker / high / medium / low
4. Evidence level: verified / hypothesis / speculation
5. Se non trovi nulla di utile, scrivi 'DISCARD: nothing actionable'
6. Massima onesta'. Massima severita'. Zero compiacenza.

## COMPETITOR
- Tinkercad Circuits (Autodesk) — gratis, 3D, ampio ma generico
- Wokwi — simulatore Arduino online, veloce, no pedagogia
- Falstad — circuit sim, potente ma per adulti
- PhET (Colorado) — simulazioni fisiche, eccellenti ma no Arduino
- Scratch/Code.org — coding ma no elettronica

## DIFFERENZIATORI ELAB
- Curriculum italiano specifico per Indicazioni Nazionali
- AI tutor Galileo con tono pedagogico per bambini
- Volumi progressivi (dall'interruttore al Simon Game)
- Simulazione + codice Arduino nello stesso ambiente
"""

print(f'program.md caricato ({len(PROGRAM_MD)} chars)')

In [ ]:
#@title 4. Research Agenda — 12 Topic rotativi

RESEARCH_AGENDA = [
    {
        'id': 'pedagogy',
        'topic': 'Pedagogia per docenti inesperti di elettronica',
        'prompt': (
            'TASK: Trova 3 principi pedagogici concreti per insegnanti che NON sanno elettronica '
            'ma devono insegnare con un simulatore di circuiti a bambini 8-12 anni.\n'
            'SUCCESS CRITERIA: Ogni principio deve essere traducibile in una feature o un prompt di Galileo.\n'
        ),
    },
    {
        'id': 'competitor',
        'topic': 'Analisi competitor EdTech simulatori 2026',
        'prompt': (
            'TASK: Confronta ELAB Tutor con Tinkercad Circuits e Wokwi. '
            'Trova 2 cose che fanno meglio di ELAB e 2 gap che nessuno copre.\n'
            'SUCCESS CRITERIA: Ogni gap deve essere un potential differenziatore per ELAB.\n'
        ),
    },
    {
        'id': 'ux_children',
        'topic': 'UX per bambini 8-12 su tablet scolastici',
        'prompt': (
            'TASK: Identifica 3 errori UX comuni nelle app educative per bambini su tablet.\n'
            'SUCCESS CRITERIA: Ogni errore deve avere un test verificabile su ELAB (font, touch, layout).\n'
        ),
    },
    {
        'id': 'ai_tutoring',
        'topic': 'AI tutoring: scaffolding e ZPD per STEM',
        'prompt': (
            'TASK: Come deve comportarsi un AI tutor (Galileo) quando un bambino sbaglia un circuito?\n'
            'Usa i concetti di ZPD (Vygotsky) e scaffolding. Galileo e un libro intelligente, NON un professore.\n'
            'SUCCESS CRITERIA: 3 pattern di risposta concreti (con esempio di frase).\n'
        ),
    },
    {
        'id': 'lim_classroom',
        'topic': 'Uso della LIM nella didattica tecnologia',
        'prompt': (
            'TASK: Descrivi il workflow di una lezione tipo su LIM (65 pollici) con un simulatore di circuiti.\n'
            'Il docente ha 1 ora. 25 alunni. Connessione instabile.\n'
            'SUCCESS CRITERIA: Identifica 2 vincoli tecnici che ELAB deve rispettare per funzionare su LIM.\n'
        ),
    },
    {
        'id': 'circuit_accuracy',
        'topic': 'Accuratezza simulatori circuiti educativi',
        'prompt': (
            'TASK: Quali sono i 3 edge case piu critici in un solver MNA/KCL per circuiti educativi?\n'
            'Contesto: ELAB ha un solver proprietario (~1700 LOC) che gestisce serie, parallelo, LED, motori.\n'
            'SUCCESS CRITERIA: Ogni edge case deve avere uno scenario di test concreto.\n'
        ),
    },
    {
        'id': 'gdpr_school',
        'topic': 'GDPR e privacy per software educativo italiano',
        'prompt': (
            'TASK: Quali sono i 3 requisiti GDPR piu importanti per un software educativo usato nelle scuole italiane?\n'
            'ELAB raccoglie: email docente, progresso studente (anonimo), chat con AI.\n'
            'SUCCESS CRITERIA: Ogni requisito deve avere una checklist verificabile.\n'
        ),
    },
    {
        'id': 'gamification',
        'topic': 'Gamification STEM per engagement bambini',
        'prompt': (
            'TASK: ELAB ha 4 giochi didattici (Trova il Guasto, Prevedi e Spiega, Circuito Misterioso, Controlla Circuito).\n'
            'Quali meccaniche di gamification mancano per aumentare engagement senza distrarre dal learning?\n'
            'SUCCESS CRITERIA: 2 meccaniche concrete con impatto stimato su retention.\n'
        ),
    },
    {
        'id': 'accessibility',
        'topic': 'Accessibilita WCAG per EdTech',
        'prompt': (
            'TASK: Quali sono i 3 problemi di accessibilita piu comuni nei simulatori di circuiti web?\n'
            'ELAB ha: canvas SVG, drag-and-drop, color-coded wires, audio feedback.\n'
            'SUCCESS CRITERIA: Ogni problema deve avere una soluzione implementabile in React.\n'
        ),
    },
    {
        'id': 'voice_nlu',
        'topic': 'Voice UI e NLU per simulatori educativi',
        'prompt': (
            'TASK: Ha senso aggiungere controllo vocale a un simulatore di circuiti per bambini?\n'
            'Se si, quali 3 comandi vocali sarebbero piu utili? Se no, perche?\n'
            'SUCCESS CRITERIA: Decisione chiara (si/no) con motivazione e evidence level.\n'
        ),
    },
    {
        'id': 'arduino_edu',
        'topic': 'Arduino nella didattica scuole medie italiane',
        'prompt': (
            'TASK: Quante scuole medie italiane usano Arduino nelle lezioni di tecnologia?\n'
            'Quali ostacoli incontrano? ELAB puo risolvere questi ostacoli?\n'
            'SUCCESS CRITERIA: Almeno 1 dato quantitativo e 2 ostacoli con soluzione ELAB.\n'
        ),
    },
    {
        'id': 'agent_memory',
        'topic': 'Memoria e contesto per agenti AI autonomi',
        'prompt': (
            'TASK: Come deve gestire la memoria un agente AI che migliora un prodotto software in loop?\n'
            'Contesto: ELAB ha un orchestratore con 9 layer di contesto ma i feedback loop sono rotti.\n'
            'SUCCESS CRITERIA: 2 pattern di memoria concreti (es. Graphiti, Mem0) con pro/contro per ELAB.\n'
        ),
    },
]

print(f'{len(RESEARCH_AGENDA)} topic di ricerca configurati')

In [ ]:
#@title 5. Structured Prompt Builder (Anatomy of a Prompt)

def build_research_prompt(agenda_item: dict, cycle: int, context: str = '') -> str:
    """Build a structured prompt following the Anatomy of a Claude Prompt pattern."""
    return f"""I want to RESEARCH '{agenda_item['topic']}' so that ELAB Tutor improves concretely.

First, read this context completely before responding:

CONTEXT:
{PROGRAM_MD[:2000]}

{f'PREVIOUS FINDINGS:{chr(10)}{context}' if context else ''}

Here is the specific research task:

{agenda_item['prompt']}

REFERENCE:
- ELAB Tutor = simulatore circuiti + AI tutor per bambini 8-12
- Principio Zero: l'insegnante inesperto e' il vero utente
- Competitor: Tinkercad, Wokwi, Falstad, PhET

SUCCESS BRIEF:
- Type: Research finding (200-400 parole, italiano)
- Recipient reaction: 'Questo insight migliora ELAB concretamente'
- Does NOT sound like: generico, vago, senza evidenze, accademico
- Success means: produce un task azionabile O una decisione informata

RULES:
1. Principio Zero sempre attivo
2. Niente claim senza evidenza
3. Severity obbligatoria (blocker/high/medium/low)
4. Evidence level obbligatorio (verified/hypothesis/speculation)
5. Se non trovi nulla di utile, scrivi 'DISCARD: nothing actionable'

OUTPUT OBBLIGATORIO (usa ESATTAMENTE questo formato):
FINDING: [3-5 frasi concrete]
EVIDENCE: [verified/hypothesis/speculation]
SEVERITY: [blocker/high/medium/low]
ACTION: [task concreto per l'orchestratore, oppure 'none']
COV: [1 frase: 'Ho verificato che...']
VERDICT: [KEEP/DISCARD]
"""


def build_judge_prompt(finding: str, topic: str) -> str:
    """Build prompt for DeepSeek to judge a finding."""
    return f"""Sei un giudice severo di ricerca EdTech. Valuta questo finding:

TOPIC: {topic}
FINDING: {finding[:1000]}

Criteri:
1. E' azionabile? (si produce un task concreto)
2. E' basato su evidenze? (non opinioni vaghe)
3. E' rilevante per ELAB Tutor? (bambini 8-12, scuole italiane)
4. Rispetta il Principio Zero? (docente inesperto e' il vero utente)

Rispondi SOLO con:
SCORE: [1-10]
VERDICT: [KEEP/DISCARD]
REASON: [1 frase]
"""


print('Prompt builder pronto.')

In [ ]:
#@title 6. results.tsv — Log degli esperimenti (come Karpathy)

RESULTS_FILE = Path('results.tsv')
KNOWLEDGE_DIR = Path('knowledge')
KNOWLEDGE_DIR.mkdir(exist_ok=True)

def init_results():
    """Initialize results.tsv with header."""
    if not RESULTS_FILE.exists():
        RESULTS_FILE.write_text(
            'cycle\ttopic_id\tseverity\tevidence\tscore\tverdict\telapsed_s\ttimestamp\n'
        )
        print('results.tsv inizializzato')
    else:
        lines = RESULTS_FILE.read_text().strip().splitlines()
        print(f'results.tsv esistente: {len(lines)-1} esperimenti')


def log_result(cycle, topic_id, severity, evidence, score, verdict, elapsed):
    """Append one experiment result."""
    timestamp = datetime.now().strftime('%Y-%m-%d %H:%M')
    line = f'{cycle}\t{topic_id}\t{severity}\t{evidence}\t{score}\t{verdict}\t{elapsed:.0f}\t{timestamp}\n'
    with open(RESULTS_FILE, 'a') as f:
        f.write(line)


def get_stats():
    """Get summary stats from results.tsv."""
    if not RESULTS_FILE.exists():
        return {'total': 0, 'kept': 0, 'discarded': 0}
    lines = RESULTS_FILE.read_text().strip().splitlines()[1:]  # skip header
    kept = sum(1 for l in lines if 'KEEP' in l)
    discarded = sum(1 for l in lines if 'DISCARD' in l)
    return {'total': len(lines), 'kept': kept, 'discarded': discarded}


init_results()

In [ ]:
#@title 7. IL LOOP AUTORESEARCH (cuore del sistema)

SYNC_EVERY_N = 10  # Auto-sync to Drive every N cycles


def _auto_sync_to_drive():
    """Auto-sync results to Google Drive (if mounted)."""
    drive_dir = Path('/content/drive/MyDrive/ELAB-Autoresearch')
    if not drive_dir.exists():
        return  # Drive not mounted
    try:
        import shutil
        if RESULTS_FILE.exists():
            shutil.copy2(RESULTS_FILE, drive_dir / 'results.tsv')
        knowledge_sync = drive_dir / 'knowledge'
        knowledge_sync.mkdir(exist_ok=True)
        for f in KNOWLEDGE_DIR.glob('*.md'):
            shutil.copy2(f, knowledge_sync / f.name)
        # Write orchestrator bridge file
        findings = []
        for f in sorted(KNOWLEDGE_DIR.glob('*.md'))[-10:]:
            findings.append({'file': f.name, 'preview': f.read_text()[:300], 'timestamp': datetime.now().isoformat()})
        (drive_dir / 'parallel-research.json').write_text(json.dumps(
            {'findings': findings, 'stats': get_stats(), 'synced_at': datetime.now().isoformat()},
            indent=2, ensure_ascii=False))
    except Exception as e:
        print(f'    Auto-sync error: {e}')


def run_autoresearch(max_iterations=MAX_ITERATIONS):
    """Loop autoresearch stile Karpathy.
    
    Per ogni iterazione:
    1. Scegli topic (rotazione ciclica)
    2. Costruisci prompt strutturato (Anatomy of a Prompt)
    3. Chiedi a Kimi K2.5 (con retry)
    4. Parse robusto dell'output (regex, tollerante a markdown)
    5. Giudica con DeepSeek-chat (veloce, ogni 3 cicli)
    6. Keep/Discard (severity >= medium AND evidence != speculation)
    7. Log in results.tsv + salva in knowledge/
    8. Auto-sync su Drive ogni 10 cicli
    9. Non fermarti mai.
    """
    
    print('=' * 60)
    print(' ELAB AUTORESEARCH - Loop Parallelo Kimi K2.5')
    print(f' Max iterations: {max_iterations}')
    print(f' Budget per iteration: {SECONDS_PER_ITERATION}s')
    print(f' Auto-sync every: {SYNC_EVERY_N} cycles')
    print('=' * 60)
    
    # Load previous context
    prev_findings = ''
    prev_files = sorted(KNOWLEDGE_DIR.glob('*.md'))
    if prev_files:
        for f in prev_files[-3:]:
            prev_findings += f.read_text()[:500] + '\n---\n'
    
    for cycle in range(1, max_iterations + 1):
        start = time.time()
        topic_idx = (cycle - 1) % len(RESEARCH_AGENDA)
        agenda = RESEARCH_AGENDA[topic_idx]
        
        print(f'\n--- Cycle {cycle}/{max_iterations} | Topic: {agenda["id"]} ---')
        
        # Step 1: Build prompt
        prompt = build_research_prompt(agenda, cycle, prev_findings)
        
        # Step 2: Call Kimi (with retry)
        print(f'  Asking Kimi K2.5...')
        finding = call_kimi(prompt, max_tokens=1500)
        
        if finding.startswith('['):
            print(f'  {finding}')
            log_result(cycle, agenda['id'], 'none', 'none', 0, 'ERROR', time.time()-start)
            continue
        
        # Step 3: Robust parse (handles markdown, extra spaces, etc.)
        parsed = parse_structured_output(finding)
        severity = parsed['severity']
        evidence = parsed['evidence']
        verdict = parsed['verdict']
        action = parsed['action']
        
        # Step 4: Judge with DeepSeek-chat (fast, every 3rd cycle)
        score = 5
        if cycle % 3 == 0 and os.environ.get('DEEPSEEK_API_KEY'):
            print(f'  Judging with DeepSeek...')
            judge_result = call_deepseek(build_judge_prompt(finding, agenda['topic']), max_tokens=200)
            for line in judge_result.splitlines():
                if 'SCORE' in line.upper():
                    nums = re.findall(r'\d+', line)
                    if nums:
                        score = min(int(nums[0]), 10)
                if 'DISCARD' in line.upper():
                    verdict = 'DISCARD'
        
        # Step 5: Keep/Discard
        should_keep = (
            verdict == 'KEEP' and
            severity in ('blocker', 'high', 'medium') and
            evidence != 'speculation'
        )
        final_verdict = 'KEEP' if should_keep else 'DISCARD'
        elapsed = time.time() - start
        
        # Step 6: Log
        log_result(cycle, agenda['id'], severity, evidence, score, final_verdict, elapsed)
        
        # Step 7: Save to knowledge/
        if should_keep:
            kf = KNOWLEDGE_DIR / f'cycle-{cycle:03d}-{agenda["id"]}.md'
            kf.write_text(
                f'# Autoresearch Cycle {cycle} - {agenda["topic"]}\n'
                f'Date: {datetime.now().isoformat()}\n'
                f'Severity: {severity} | Evidence: {evidence} | Score: {score}\n\n'
                f'{finding}\n\n---\nAction: {action}\n'
            )
            prev_findings = finding[:500] + '\n---\n' + prev_findings[:1000]
        
        # Step 8: Auto-sync to Drive
        if cycle % SYNC_EVERY_N == 0:
            print(f'  Auto-syncing to Drive...')
            _auto_sync_to_drive()
        
        # Status
        stats = get_stats()
        tag = 'KEEP' if should_keep else 'DISCARD'
        print(f'  [{tag}] severity={severity} evidence={evidence} score={score} ({elapsed:.0f}s)')
        print(f'  Stats: {stats["total"]} total, {stats["kept"]} kept, {stats["discarded"]} discarded')
        if should_keep and action != 'none':
            print(f'  ACTION: {action[:100]}')
        
        # Rate limit
        remaining = SECONDS_PER_ITERATION - elapsed
        if remaining > 0:
            time.sleep(remaining)
    
    # Final sync + report
    _auto_sync_to_drive()
    stats = get_stats()
    print('\n' + '=' * 60)
    print(f' AUTORESEARCH COMPLETED')
    print(f' Total: {stats["total"]} | Kept: {stats["kept"]} | Discarded: {stats["discarded"]}')
    print(f' Keep rate: {stats["kept"]/max(stats["total"],1)*100:.0f}%')
    print('=' * 60)


print('Loop pronto. Esegui cella 8 per avviarlo.')

In [ ]:
#@title 8. AVVIA IL LOOP
#@markdown Esegui questa cella per avviare il loop autoresearch.
#@markdown Il loop gira finche' non lo interrompi manualmente (Ctrl+C o stop cella).

run_autoresearch(max_iterations=MAX_ITERATIONS)

In [ ]:
#@title 9. Sync con Google Drive (opzionale)

if SYNC_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    
    import shutil
    sync_dir = Path('/content/drive/MyDrive/ELAB-Autoresearch')
    sync_dir.mkdir(parents=True, exist_ok=True)
    
    # Copy results
    if RESULTS_FILE.exists():
        shutil.copy2(RESULTS_FILE, sync_dir / 'results.tsv')
    
    # Copy knowledge
    sync_knowledge = sync_dir / 'knowledge'
    sync_knowledge.mkdir(exist_ok=True)
    for f in KNOWLEDGE_DIR.glob('*.md'):
        shutil.copy2(f, sync_knowledge / f.name)
    
    # Write parallel-research.json for orchestrator
    findings_for_orchestrator = []
    for f in sorted(KNOWLEDGE_DIR.glob('*.md'))[-10:]:
        findings_for_orchestrator.append({
            'file': f.name,
            'preview': f.read_text()[:300],
            'timestamp': datetime.now().isoformat(),
        })
    
    sync_json = sync_dir / 'parallel-research.json'
    sync_json.write_text(json.dumps({
        'findings': findings_for_orchestrator,
        'stats': get_stats(),
        'synced_at': datetime.now().isoformat(),
    }, indent=2, ensure_ascii=False))
    
    print(f'Sincronizzato in {sync_dir}')
    print(f'  results.tsv: {RESULTS_FILE.stat().st_size} bytes')
    print(f'  knowledge/: {len(list(KNOWLEDGE_DIR.glob("*.md")))} file')
    print(f'  parallel-research.json: pronto per orchestratore')

In [ ]:
#@title 10. Dashboard — Visualizza risultati

if RESULTS_FILE.exists():
    lines = RESULTS_FILE.read_text().strip().splitlines()
    print(f'RISULTATI AUTORESEARCH ({len(lines)-1} esperimenti)')
    print('=' * 80)
    
    # Parse TSV
    header = lines[0].split('\t')
    print(f'{"|":^5}'.join(header))
    print('-' * 80)
    
    for line in lines[1:]:
        cols = line.split('\t')
        verdict = cols[5] if len(cols) > 5 else '?'
        emoji = 'O' if verdict.strip() == 'KEEP' else 'X'
        print(f'[{emoji}] {line}')
    
    stats = get_stats()
    print('\n' + '=' * 80)
    print(f'Keep rate: {stats["kept"]}/{stats["total"]} ({stats["kept"]/max(stats["total"],1)*100:.0f}%)')
    
    # Topic breakdown
    topic_counts = {}
    for line in lines[1:]:
        cols = line.split('\t')
        tid = cols[1] if len(cols) > 1 else '?'
        verdict = cols[5].strip() if len(cols) > 5 else '?'
        topic_counts.setdefault(tid, {'keep': 0, 'discard': 0})
        if verdict == 'KEEP':
            topic_counts[tid]['keep'] += 1
        else:
            topic_counts[tid]['discard'] += 1
    
    print('\nPer topic:')
    for tid, counts in sorted(topic_counts.items()):
        total = counts['keep'] + counts['discard']
        rate = counts['keep'] / max(total, 1) * 100
        print(f'  {tid:20s}: {counts["keep"]} kept / {total} total ({rate:.0f}%)')
else:
    print('Nessun risultato ancora. Avvia il loop (cella 8).')